In [0]:
import requests
import json
from datetime import datetime, timedelta
import pandas as pd

In [0]:
volume_path = "/Volumes/fhir_api_data_catalog/raw/api_data"

base_url='https://hapi.fhir.org/baseR4'
api_objects = ["Patient","Encounter","Observation","Condition"]

headers = {
    "Content-Type": "application/json"
}

today = datetime.now()
target_dates = [(today - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(3)]

In [0]:
for api_object in api_objects:
    print(f'----------Fetching {api_object} data---------------')
    for fetch_date in target_dates:
        print(f"-----------------Processing {api_object} data from {fetch_date}-----------------------")
        folder_path = f"{volume_path}/{api_object}/date={fetch_date}"
        url = f"{base_url}/{api_object}?_lastUpdated={fetch_date}"
        page=1

        while url:
            try: 
                response=requests.get(url)
                response.raise_for_status()
                data = response.json()
                entries = data.get("entry", [])

                if not entries:
                    print(f"No more records found for {api_object} on {fetch_date}.")
                    break

                records = [entry["resource"] for entry in entries if "resource" in entry]

                file_name = f"page_{page}.json"
                file_path = f"{folder_path}/{file_name}"

                dbutils.fs.put(file_path, json.dumps(records), overwrite=True)
                print(f"Saved {len(records)} records to: {file_path}")

                links = data.get("link", [])
                
                # Search for the link where relation == "next"
                url = None
                for link in links:
                    if link.get("relation") == "next":
                        url = link.get("url")
                        break
                        
                page += 1

                if page > 50:
                    print(f"---------------------Stopping after 100 pages for {api_object} on {fetch_date}------------------")
                    break
                
            except Exception as e:
                print(f"Error fetching {api_object} on {fetch_date} (Page {page}): {e}")
                break

    log_metadata = {
    "run_timestamp": datetime.now(),
    "pipeline_step": "REST_API_INGESTION",
    "target_catalog": "fhir_api_data_catalog",
    "status": "SUCCESS",
    # You can expand this to include variables tracked in your loop
    "objects_processed": str(api_object)
    }

    df_audit_log = spark.createDataFrame(pd.DataFrame([log_metadata]))

    # Append it to a permanent Delta table
    df_audit_log.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("fhir_api_data_catalog.raw.pipeline_audit_log")

    print(f"Metadata successfully logged {api_object} to fhir_api_data_catalog.raw.pipeline_audit_log")

            

----------Fetching Patient data---------------
-----------------Processing Patient data from 2026-08-09-----------------------
Wrote 13931 bytes.
Saved 20 records to: /Volumes/fhir_api_data_catalog/raw/api_data/Patient/date=2026-08-09/page_1.json
Wrote 12869 bytes.
Saved 20 records to: /Volumes/fhir_api_data_catalog/raw/api_data/Patient/date=2026-08-09/page_2.json
Wrote 11880 bytes.
Saved 20 records to: /Volumes/fhir_api_data_catalog/raw/api_data/Patient/date=2026-08-09/page_3.json
Wrote 12512 bytes.
Saved 20 records to: /Volumes/fhir_api_data_catalog/raw/api_data/Patient/date=2026-08-09/page_4.json
Wrote 4096 bytes.
Saved 5 records to: /Volumes/fhir_api_data_catalog/raw/api_data/Patient/date=2026-08-09/page_5.json
-----------------Processing Patient data from 2026-08-08-----------------------
Wrote 17159 bytes.
Saved 20 records to: /Volumes/fhir_api_data_catalog/raw/api_data/Patient/date=2026-08-08/page_1.json
Wrote 18106 bytes.
Saved 20 records to: /Volumes/fhir_api_data_catalog/raw/

In [0]:
%sql
select * from fhir_api_data_catalog.raw.pipeline_audit_log

run_timestamp,pipeline_step,target_catalog,status,objects_processed
2026-08-09T13:54:35.348Z,REST_API_INGESTION,fhir_api_data_catalog,SUCCESS,"['Patient', 'Encounter', 'Observation', 'Condition']"
2026-08-09T14:14:32.956Z,REST_API_INGESTION,fhir_api_data_catalog,SUCCESS,Observation
2026-08-09T14:13:42.402Z,REST_API_INGESTION,fhir_api_data_catalog,SUCCESS,Encounter
2026-08-09T14:14:44.304Z,REST_API_INGESTION,fhir_api_data_catalog,SUCCESS,Condition
2026-08-09T14:12:35.197Z,REST_API_INGESTION,fhir_api_data_catalog,SUCCESS,Patient
